In [0]:
# =============================================================================
# CELL 1 — Run metadata
# Receives pipeline-level IDs from parent orchestrator. Falls back to
# generating its own if run standalone (useful during development).
# =============================================================================
%load_ext autoreload
%autoreload 2


import uuid, time
from datetime import datetime, timezone
from pyspark.sql import Row
from pyspark.sql.functions import current_timestamp, lit, input_file_name, col
from pyspark.sql.types import (
    StructType, StructField, StringType, TimestampType,
    DoubleType, LongType, IntegerType
)

import sys
sys.path.append("/Workspace/Shared")

from pipeline_utils import get_notebook_context
from pipeline_logging import pipeline_log_upsert, pipeline_step_log_upsert, ingestion_log_insert, transform_detail_log_insert


# ----------------------------------------------------------------------------
# Catalog / layer constants (per project standards)
# Assumes this notebook will be called by another and passed parameters, but is also set up
# to function in isolation for dev and testing.
# ----------------------------------------------------------------------------
CATALOG   = "Vinoworld"
BRONZE    = f"{CATALOG}.bronze"
SILVER    = f"{CATALOG}.silver"
GOLD      = f"{CATALOG}.gold"
AUDIT     = f"{CATALOG}.audit"
RAW_FILES = "/Volumes/vinoworld/datafiles/"


START_TIMESTAMP = datetime.now(timezone.utc)
RUN_INT_ID      = str(uuid.uuid4())
NOTEBOOK_PATH = "/Workspace/Users/zieder0022@gmail.com/Vinoworld/000-Pipeline_Logging_test"


# Pipeline-level identifiers — shared across all notebooks in a pipeline run.
# Read from parent via widgets; fall back to standalone mode if not provided.
try:
    dbutils.widgets.text("pipeline_run_id",   "")
    dbutils.widgets.text("pipeline_start_ts", "")
    dbutils.widgets.text("pipeline_name",     "")

    PIPELINE_RUN_ID   = dbutils.widgets.get("pipeline_run_id")   or RUN_INT_ID
    PIPELINE_START_TS = dbutils.widgets.get("pipeline_start_ts") or START_TIMESTAMP.isoformat()
    PIPELINE_NAME     = dbutils.widgets.get("pipeline_name")     or "Testing Pipeline"
except Exception:
    # Not running in Databricks (e.g. local dev) — use safe defaults
    PIPELINE_RUN_ID   = f"STANDALONE_{RUN_INT_ID}"
    PIPELINE_START_TS = START_TIMESTAMP.isoformat()
    PIPELINE_NAME     = "standalone"

# Convert pipeline timestamp back to datetime for use in queries
PIPELINE_START_TS = datetime.fromisoformat(PIPELINE_START_TS)

PIPELINE_STATUS = "running"
PIPELINE_END_TS = None
ERROR_MESSAGE   = None                       # populated only on failure

print(f"PIPELINE_RUN_ID : {PIPELINE_RUN_ID}")
print(f"PIPELINE_NAME   : {PIPELINE_NAME}")
print(f"PIPELINE_START_TS   : {PIPELINE_START_TS}")
print(f"RUN_INT_ID          : {RUN_INT_ID}")
print(f"NOTEBOOK        : {NOTEBOOK_PATH}")

In [0]:
# Initialize remaining variables the represent ingestion_log columns

LAYER             = "bronze"                   # 'bronze' | 'silver' | 'gold'
SOURCE_PATH   = f"{RAW_FILES}/arancione/"                           # e.g. f"{RAW_FILES}/Arancione/"
TARGET_TABLE  = f"{BRONZE}.sales"              # overwrite per step
END_TIMESTAMP    = None                        # set on completion
DURATION_SECONDS = None                        # computed at completion
STATUS            = "running"                  # 'running' | 'success' | 'failed'
SOURCE_ROW_COUNT  = 0
TARGET_ROW_COUNT  = 0
FILES_PROCESSED   = 0




---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-7642969637231991>, line 4
      1 # Initialize remaining variables the represent ingestion_log columns
      3 LAYER             = "bronze"                   # 'bronze' | 'silver' | 'gold'
----> 4 SOURCE_PATH   = f"{RAW_FILES}/arancione/"                           # e.g. f"{RAW_FILES}/Arancione/"
      5 TARGET_TABLE  = f"{BRONZE}.sales"              # overwrite per step
      6 END_TIMESTAMP    = None                        # set on completion

NameError: name 'RAW_FILES' is not defined

In [0]:
%skip
# ----------------------------------------------------------------------------
# Schema for the audit row - enforces types when building the DataFrame
# ----------------------------------------------------------------------------
from pyspark.sql.types import Row, StructType, StructField, StringType, IntegerType, LongType, TimestampType

audit_schema = StructType([
    StructField("pipeline_run_id",   LongType(),    False),
    StructField("pipeline_name",     StringType(),    True),
    StructField("status",            StringType(),    False),
    StructField("started_timestamp",   TimestampType(), False),
    StructField("ended_timestamp",     TimestampType(), True),
    StructField("error_message",     StringType(),    True),
])



PIPELINE_STATUS = "success"
PIPELINE_END_TS = datetime.now()
ERROR_MESSAGE = "testing update"

"""
Build a single-row DataFrame from the current audit variable values.
Call this right before inserting/merging into the audit log table.
"""
row = Row(
    pipeline_run_id   = PIPELINE_RUN_ID,
    pipeline_name     = PIPELINE_NAME,
    status            = PIPELINE_STATUS,
    started_timestamp = PIPELINE_START_TS,
    ended_timestamp   = PIPELINE_END_TS,
    error_message     = ERROR_MESSAGE,
)
df_log_row = spark.createDataFrame([row], schema=audit_schema)

from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark,  "vinoworld.audit.pipeline_log")
# 

delta_table.alias("target") \
  .merge(
    df_log_row.alias("source"),
    "target.pipeline_run_id = source.pipeline_run_id"  # Your join condition
  ) \
  .whenMatchedUpdateAll() \
  .whenNotMatchedInsertAll() \
  .execute()



In [0]:
%skip
%load_ext autoreload
%autoreload 2

PIPELINE_STATUS = "running"
PIPELINE_END_TS = None 
ERROR_MESSAGE = None

# pipeline_log_upsert(spark, PIPELINE_RUN_ID, PIPELINE_NAME, PIPELINE_STATUS, PIPELINE_START_TS)

pipeline_log_upsert(spark, PIPELINE_RUN_ID, PIPELINE_NAME, PIPELINE_STATUS, PIPELINE_START_TS, PIPELINE_END_TS, ERROR_MESSAGE)

In [0]:
%sql

SELECT * FROM vinoworld.audit.pipeline_step_log



In [0]:
%load_ext autoreload
%autoreload 2

from pipeline_utils import get_notebook_context
nb = get_notebook_context(dbutils)

notebook_folder = nb['notebook_folder']
notebook_name = nb['notebook_folder']

display(nb)

print(f"notebook_folder : {notebook_folder}")
print(f"notebook_name   : {notebook_name}")



In [0]:
%skip
step_log_id       = str(uuid.uuid4())
STEP_LOG_ID = step_log_id
pipeline_run_id   = PIPELINE_RUN_ID
step_sequence     = 1
notebook_folder   = "Vinoworld"
notebook_name     = "bronze_celeste"
layer             = "bronze"
target_table      = "vinoworld.bronze.sales_celeste"
status            = "running"
rows_read         = 0
rows_written      = 0
# started_timestamp = spark.sql("SELECT current_timestamp() as ts").first()["ts"]   # datetime.now(timezone.utc)
# ended_timestamp   = spark.sql("SELECT current_timestamp() as ts").first()["ts"]
started_timestamp = datetime.now(timezone.utc)

In [0]:
%load_ext autoreload
%autoreload 2


ended_timestamp   = datetime.now(timezone.utc)
error_message     = "testing error message-jdd"

print(f"step_log_id         : {step_log_id}")
print(f"RUN_INT_ID          : {RUN_INT_ID}")

RUN_INT_ID      = str(uuid.uuid4())
print(f"RUN_INT_ID2          : {RUN_INT_ID}")

# pipeline_step_log_upsert(spark, step_log_id, pipeline_run_id, step_sequence, notebook_folder, notebook_name, status, started_timestamp, layer, target_table)


pipeline_step_log_upsert(spark, step_log_id, pipeline_run_id, step_sequence, notebook_folder, notebook_name, status, started_timestamp, layer, target_table, rows_read, rows_written,  ended_timestamp,  error_message)


In [0]:
%skip
%sql
SELECT * FROM vinoworld.audit.pipeline_step_log

In [0]:
%load_ext autoreload
%autoreload 2

STEP_LOG_ID = str(uuid.uuid4())



# Load Celeste csv files and add the source_file_path column
RAW_FILES = "/Volumes/vinoworld/datafiles/"
df = (
    spark.read.format("csv")
    .option("header", "true")
    .load(f"{RAW_FILES}celeste")
    .withColumn("source_file_path", col("_metadata.file_path"))   
)

# Create a new dataframe with only the source_file_path column
df_files = df.select("source_file_path").distinct()
# display(df_files)

 # Setup variables to pass to ingestion_log_insert
pipeline_run_id = PIPELINE_RUN_ID
step_log_id = STEP_LOG_ID
source_system = "Celeste"
target_table = "vinoworld.bronze.sales_celeste"
error_message = "testing error message-jdd"
ingestion_ts = datetime.now(timezone.utc)
# DBTITLE 1,



transform_detail_log_insert(
    spark,
    df_files,
    pipeline_run_id,
    step_log_id,
    source_system,
    target_table, 
    error_message,
    ingestion_ts)










In [0]:


# ----------------------------------------------------------
# Test setup for transform_detail_log_insert
# ----------------------------------------------------------

STEP_LOG_ID = 888

transform_started_timestamp = datetime.now(timezone.utc)
transform_ended_timestamp   = datetime.now(timezone.utc)

pipeline_run_id           = PIPELINE_RUN_ID
step_log_id               = STEP_LOG_ID          # FK to the step log created above
source_table              = "vinoworld.bronze.dim_currency"
target_table              = "vinoworld.silver.dim_currency"
status                    = "succeeded"
rows_read                 = 1250
rows_written              = 1200
rows_inserted             = 1100
rows_updated              = 100
rows_expired              = 0                     # SCD2 only, 0 for Type 1
rows_rejected             = 50
rows_deduplicated         = 10
validation_rules_applied  = '["CurrencyCode NOT NULL", "ExchangeRate > 0"]'
schema_drift_detected     = False
schema_drift_detail       = None
error_message             = None                  # None on success

print(f"pipeline_run_id         : {pipeline_run_id}")
print(f"step_log_id             : {step_log_id}")
print(f"source_table            : {source_table}")
print(f"target_table            : {target_table}")
print(f"status                  : {status}")
print(f"rows_read               : {rows_read}")
print(f"rows_written            : {rows_written}")
print(f"rows_inserted           : {rows_inserted}")
print(f"rows_updated            : {rows_updated}")
print(f"rows_expired            : {rows_expired}")
print(f"rows_rejected           : {rows_rejected}")
print(f"rows_deduplicated       : {rows_deduplicated}")
print(f"validation_rules_applied: {validation_rules_applied}")
print(f"schema_drift_detected   : {schema_drift_detected}")
print(f"schema_drift_detail     : {schema_drift_detail}")
print(f"error_message           : {error_message}")
print(f"started_timestamp       : {transform_started_timestamp}")
print(f"ended_timestamp         : {transform_ended_timestamp}")

transform_detail_log_insert(
    spark,
    pipeline_run_id           = pipeline_run_id,
    step_log_id               = step_log_id,
    source_table              = source_table,
    target_table              = target_table,
    status                    = status,
    started_timestamp         = transform_started_timestamp,
    rows_read                 = rows_read,
    rows_written              = rows_written,
    rows_inserted             = rows_inserted,
    rows_updated              = rows_updated,
    rows_expired              = rows_expired,
    rows_rejected             = rows_rejected,
    rows_deduplicated         = rows_deduplicated,
    validation_rules_applied  = validation_rules_applied,
    schema_drift_detected     = schema_drift_detected,
    schema_drift_detail       = schema_drift_detail,
    error_message             = error_message,
    ended_timestamp           = transform_ended_timestamp,
)

In [0]:
%sql
-- truncate table vinoworld.audit.ingestion_log;

-- select * from vinoworld.audit.pipeline_log;
-- select * from vinoworld.audit.pipeline_step_log;
select * from vinoworld.audit.transform_detail_log;
-- select * from vinoworld.audit.ingestion_log;

In [0]:


testval =  uuid.uuid4().int >> 64
print(testval)